# Introduction to CUDA Kernels
### GPU Programming Fundamentals —  Agentic AI & AI Engineering Series
### Author: Abhishek

**Runs on:** Google Colab, free T4 GPU. No local install needed — just
`Runtime -> Change runtime type -> T4 GPU` before you start.

---

## Why this session exists

In the agent harness session, every call to `call_model()` on your Ollama
or Hugging Face backend was, underneath, a GPU doing thousands of tiny
identical arithmetic operations at once. That's not a metaphor — it's
literally what a CUDA kernel is. This session opens that black box.

You don't need a C++ background for this. We'll write real GPU code in
plain Python first (using Numba), see it run faster than your CPU, and only
*read* — never write — the equivalent raw CUDA C, so vendor documentation
and Stack Overflow snippets stop looking like an alien language.

## The mental model, in one picture

Think of your CPU as **one extremely skilled senior engineer** who can
handle any ticket, one at a time, very fast and very flexibly.

Think of your GPU as **ten thousand junior associates**, each of whom can
only do one very simple, identical step — but they all do it *at the same
time*. A GPU is useless for "read this ticket and decide what to do"
(that's CPU/LLM-reasoning work). A GPU is extraordinary for "multiply these
two million numbers together, all at once" — which is exactly what powers
every layer of a neural network.

**A CUDA kernel is the instruction you hand to all ten thousand associates
at once.** That's the whole idea. Everything else in this notebook is
detail on top of that one sentence.


## 0. Setup: confirm you have a GPU

In [ ]:
!nvidia-smi


If that printed a table with a GPU name (something like "Tesla T4"), you're
set. If it says "command not found" or shows no GPU, go to
**Runtime → Change runtime type → Hardware accelerator → GPU** (or "T4 GPU")
and re-run this cell.


In [ ]:
!pip install -q numba


In [ ]:
from numba import cuda
import numpy as np
import time

print("CUDA available to Numba:", cuda.is_available())
print(cuda.get_current_device().name.decode())


## 1. Threads, blocks, and grids — the three words you need

Every GPU program organizes its ten thousand associates into a strict
hierarchy:

| Term | What it is | Analogy |
|---|---|---|
| **Thread** | One associate, running one instance of your instruction | One team member |
| **Block** | A group of threads that can share a small fast scratchpad and coordinate | One team |
| **Grid** | All the blocks launched for this job | The whole department, all teams |

You write **one function** — called a **kernel** — describing what a single
thread does. You then tell the GPU "run this kernel across a grid of
`N` blocks, each with `M` threads." The GPU runs your one function that
many times, in parallel, with each thread able to ask "which one am I?"
and act on a different piece of data accordingly.

That question — "which one am I?" — is answered by three built-in values
every kernel can read:

- `threadIdx` — my position within my block
- `blockIdx` — my block's position within the grid
- `blockDim` — how many threads are in each block

Combine them and every single thread across the entire GPU gets a unique
index. That index is how a thread knows *which row of data is mine*.


## 2. Your first kernel: adding two arrays, in plain Python

This is the "hello world" of GPU programming. We'll add two arrays of a
million numbers together — element by element — using Numba's `@cuda.jit`
decorator, which compiles ordinary-looking Python into a real GPU kernel.


In [ ]:
@cuda.jit
def add_kernel(a, b, result):
    """Runs once per thread. Each thread adds exactly one pair of numbers."""
    i = cuda.grid(1)  # my unique global index, computed from blockIdx/threadIdx/blockDim for us
    if i < result.size:  # guard against threads that don't map to real data
        result[i] = a[i] + b[i]


N = 1_000_000
a = np.random.rand(N).astype(np.float32)
b = np.random.rand(N).astype(np.float32)

# Copy data to the GPU's own memory - the GPU cannot touch your regular RAM directly
a_gpu = cuda.to_device(a)
b_gpu = cuda.to_device(b)
result_gpu = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # round up so every element is covered

add_kernel[blocks_per_grid, threads_per_block](a_gpu, b_gpu, result_gpu)

result = result_gpu.copy_to_host()  # bring the answer back to regular RAM
print("Correct:", np.allclose(result, a + b))
print(f"Launched {blocks_per_grid} blocks x {threads_per_block} threads = {blocks_per_grid * threads_per_block:,} threads for {N:,} elements")


Read that launch line again: `add_kernel[blocks_per_grid, threads_per_block](...)`.
That square-bracket syntax is Numba's way of saying "launch this many
blocks, this many threads per block" — the grid configuration sits right
next to the function call. This is the one piece of syntax in this whole
notebook that doesn't look like ordinary Python, and now you know exactly
what it means.


## 3. CPU vs GPU: when is this actually worth it?

Let's time it properly. This section matters as much as the last one —
knowing when *not* to reach for a GPU is a real engineering judgment call,
not a footnote.


In [ ]:
def time_cpu_add(n, repeats=5):
    a = np.random.rand(n).astype(np.float32)
    b = np.random.rand(n).astype(np.float32)
    start = time.perf_counter()
    for _ in range(repeats):
        result = a + b
    return (time.perf_counter() - start) / repeats


def time_gpu_add(n, repeats=5):
    a = np.random.rand(n).astype(np.float32)
    b = np.random.rand(n).astype(np.float32)
    a_gpu = cuda.to_device(a)
    b_gpu = cuda.to_device(b)
    result_gpu = cuda.device_array(n, dtype=np.float32)
    threads_per_block = 256
    blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

    add_kernel[blocks_per_grid, threads_per_block](a_gpu, b_gpu, result_gpu)  # warm-up: first call compiles the kernel
    cuda.synchronize()

    start = time.perf_counter()
    for _ in range(repeats):
        add_kernel[blocks_per_grid, threads_per_block](a_gpu, b_gpu, result_gpu)
    cuda.synchronize()  # wait for the GPU to actually finish before stopping the clock
    return (time.perf_counter() - start) / repeats


print(f"{'Size':>12} | {'CPU (ms)':>10} | {'GPU (ms)':>10} | {'Speedup':>8}")
print("-" * 50)
for n in [1_000, 100_000, 10_000_000, 100_000_000]:
    cpu_time = time_cpu_add(n) * 1000
    gpu_time = time_gpu_add(n) * 1000
    print(f"{n:>12,} | {cpu_time:>10.4f} | {gpu_time:>10.4f} | {cpu_time / gpu_time:>7.2f}x")


Look at the small sizes first. **The GPU is often slower for small arrays.**
This isn't a bug — every GPU launch has fixed overhead (copying data across
the PCIe bus, scheduling the kernel) that a CPU simply doesn't pay. The GPU
only wins once there's enough raw parallel work to justify that overhead.

This is the exact same judgment call as deciding whether a task is worth
spinning up an RPA bot for versus just doing it by hand: **the tool has a
fixed cost, and it only pays off past a certain scale.**


## 4. A meatier example: matrix multiplication

Vector addition barely uses the GPU's actual strength: doing real
*compute*, not just moving numbers around. Matrix multiplication is the
single most important operation in deep learning — every layer of every
neural network, including the local LLMs you ran in Ollama, is dominated
by matrix multiplies. Let's write one.


In [ ]:
@cuda.jit
def matmul_kernel(A, B, C):
    """Each thread computes exactly one element of the output matrix C = A @ B."""
    row, col = cuda.grid(2)  # a 2D index this time - one for the row, one for the column
    if row < C.shape[0] and col < C.shape[1]:
        total = 0.0
        for k in range(A.shape[1]):
            total += A[row, k] * B[k, col]
        C[row, col] = total


N = 512  # a 512x512 matrix multiply
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

A_gpu = cuda.to_device(A)
B_gpu = cuda.to_device(B)
C_gpu = cuda.device_array((N, N), dtype=np.float32)

threads_per_block = (16, 16)  # a 16x16 tile of threads per block - a common, reasonable default
blocks_per_grid = (
    (N + threads_per_block[0] - 1) // threads_per_block[0],
    (N + threads_per_block[1] - 1) // threads_per_block[1],
)

matmul_kernel[blocks_per_grid, threads_per_block](A_gpu, B_gpu, C_gpu)
C = C_gpu.copy_to_host()

print("Correct:", np.allclose(C, A @ B, atol=1e-2))
print(f"Grid: {blocks_per_grid} blocks of {threads_per_block} threads = {blocks_per_grid[0]*blocks_per_grid[1]*256:,} threads total")


In [ ]:
def time_cpu_matmul(n, repeats=3):
    A = np.random.rand(n, n).astype(np.float32)
    B = np.random.rand(n, n).astype(np.float32)
    start = time.perf_counter()
    for _ in range(repeats):
        C = A @ B
    return (time.perf_counter() - start) / repeats


def time_gpu_matmul(n, repeats=3):
    A = np.random.rand(n, n).astype(np.float32)
    B = np.random.rand(n, n).astype(np.float32)
    A_gpu, B_gpu = cuda.to_device(A), cuda.to_device(B)
    C_gpu = cuda.device_array((n, n), dtype=np.float32)
    tpb = (16, 16)
    bpg = ((n + 15) // 16, (n + 15) // 16)

    matmul_kernel[bpg, tpb](A_gpu, B_gpu, C_gpu)  # warm-up
    cuda.synchronize()

    start = time.perf_counter()
    for _ in range(repeats):
        matmul_kernel[bpg, tpb](A_gpu, B_gpu, C_gpu)
    cuda.synchronize()
    return (time.perf_counter() - start) / repeats


print(f"{'Size':>8} | {'CPU (ms)':>10} | {'GPU (ms)':>10} | {'Speedup':>8}")
print("-" * 46)
for n in [128, 512, 1024]:
    cpu_time = time_cpu_matmul(n) * 1000
    gpu_time = time_gpu_matmul(n) * 1000
    print(f"{n:>8} | {cpu_time:>10.3f} | {gpu_time:>10.3f} | {cpu_time / gpu_time:>7.2f}x")


Now the GPU wins clearly, and the margin grows with size — matrix
multiplication does `N` multiply-adds per output element, so the amount of
real work scales much faster than the fixed launch overhead. This is
exactly why GPUs transformed deep learning: a transformer model is,
overwhelmingly, a very long sequence of matrix multiplications.


## 5. Reading raw CUDA C (you will never need to write this)

Numba compiles the Python you just wrote down into the same machine code a
C++ CUDA program would produce. It's worth seeing the "native" version once
so that when you hit a CUDA snippet in vendor docs, a GitHub repo, or a
Stack Overflow answer, it reads as familiar rather than as a wall of
symbols. You are not expected to write this yourself.


In [ ]:
!pip install -q nvcc4jupyter
%load_ext nvcc4jupyter


In [ ]:
%%cuda
#include <stdio.h>

// This is the exact same idea as add_kernel above, in the language the GPU vendor (NVIDIA) documents in.
// __global__ marks a function as a kernel: callable from the CPU, run on the GPU.
__global__ void add_kernel(float *a, float *b, float *result, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;  // the exact same index math cuda.grid(1) did for us in Numba
    if (i < n) {
        result[i] = a[i] + b[i];
    }
}

int main() {
    int n = 10;
    float a[10], b[10], result[10];
    for (int i = 0; i < n; i++) { a[i] = i; b[i] = i * 2; }

    float *a_gpu, *b_gpu, *result_gpu;
    cudaMalloc(&a_gpu, n * sizeof(float));   // the C equivalent of cuda.to_device()
    cudaMalloc(&b_gpu, n * sizeof(float));
    cudaMalloc(&result_gpu, n * sizeof(float));

    cudaMemcpy(a_gpu, a, n * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(b_gpu, b, n * sizeof(float), cudaMemcpyHostToDevice);

    add_kernel<<<1, 10>>>(a_gpu, b_gpu, result_gpu, n);  // the <<<blocks, threads>>> launch syntax

    cudaMemcpy(result, result_gpu, n * sizeof(float), cudaMemcpyDeviceToHost);  // the C equivalent of copy_to_host()

    for (int i = 0; i < n; i++) {
        printf("%.0f + %.0f = %.0f\n", a[i], b[i], result[i]);
    }
    return 0;
}


Put the two side by side:

| Numba (Python) | Raw CUDA C | Meaning |
|---|---|---|
| `@cuda.jit` | `__global__` | marks this as a kernel |
| `cuda.grid(1)` | `blockIdx.x * blockDim.x + threadIdx.x` | "which thread am I, globally?" |
| `cuda.to_device(a)` | `cudaMalloc` + `cudaMemcpy(...HostToDevice)` | copy data to the GPU |
| `result_gpu.copy_to_host()` | `cudaMemcpy(...DeviceToHost)` | copy the answer back |
| `kernel[blocks, threads](...)` | `kernel<<<blocks, threads>>>(...)` | launch the kernel |

Same five ideas, different spelling. That's genuinely the whole gap between
"I can write GPU code in Python" and "I can read a CUDA C repo."


## 6. One more idea worth knowing: not all GPU memory is equal

Everything above used **global memory** — the GPU's large main memory,
which every thread can read and write, but which is relatively slow to
access. There's a second, much smaller and much faster kind called
**shared memory**, private to each block, that threads in the same block
can use to avoid repeatedly fetching the same data from global memory.

The warehouse analogy: global memory is the warehouse shelf, a bit of a
walk away; shared memory is the workbench right in front of you, holding
only what your immediate team needs for the task at hand. Real
high-performance kernels (including the ones inside PyTorch and Ollama's
inference engine) spend a lot of their design effort minimizing trips to
the warehouse. This is genuinely deep-dive territory — flagged here so the
term doesn't come as a surprise later, not something to implement today.


## 7. Tying it back to the agent harness session

Every time your harness called `call_model()` against a local Ollama
model, here's what actually happened underneath, in outline:

1. Your prompt was tokenized and turned into numbers
2. Those numbers flowed through dozens of transformer layers
3. **Each layer is dominated by matrix multiplications** — exactly the
   kernel you just wrote and benchmarked in Section 4, just much larger
   and repeated many times
4. The GPU ran thousands of threads at once to compute each layer's output
5. The final numbers were turned back into the words you saw

A 3-billion-parameter model like `llama3.2:3b` means roughly 3 billion
numbers get multiplied and added together for *every single token* it
generates. That workload is only tractable because of exactly the
thread/block/grid parallelism you used today. Quantization (running a
model at lower numerical precision, which you may have seen mentioned for
local models) is, underneath, a decision about exactly how much data each
of those threads has to move and how fast the multiply-adds run.

## Recap

| Concept | One-line definition |
|---|---|
| Kernel | The function that runs once per thread |
| Thread | One unit of parallel work |
| Block | A group of threads that can share fast memory |
| Grid | All the blocks launched for one kernel call |
| `cuda.grid(1)` / `blockIdx`+`threadIdx`+`blockDim` | How a thread finds out which piece of data is its own |
| Global memory | Large, shared, relatively slow GPU memory |
| Shared memory | Small, fast, per-block scratchpad |
| Launch overhead | The fixed cost of starting GPU work — why GPUs lose on small jobs |

## Exercises

1. **Change the block size.** Re-run the vector-add timing with
   `threads_per_block = 32` and then `1024`. Does the speedup change? Why
   might there be a sweet spot rather than "bigger is always better"?
2. **Write a squaring kernel.** Write a new `@cuda.jit` kernel that takes
   one array and squares every element in place. Time it against
   `numpy`'s `a ** 2` at a few sizes, the same way Section 3 did.
3. **Predict before you run.** Before re-running Section 4's matmul
   benchmark at `N = 2048`, write down your guess for the speedup. Then
   run it and see how close you were — this is the fastest way to build
   real intuition for where the crossover point sits on a given GPU.
